# Tutorial: CivicMorph Boulder Colorado Tutorial

## Audience
Urban analysts, planners, and geospatial developers who want to generate speculative, human-scale scenarios for Boulder, Colorado.

## Prerequisites
- A local clone of CivicMorph
- Python 3.11+
- Optional extras for integrations (`graph2city`, `mesa`)
- Boulder input files in `data/boulder/`

## Learning Goals
By the end of this notebook you will be able to:
1. Build Boulder baseline artifacts from local OSM/terrain files.
2. Generate and score an ensemble with optional ABM evaluation.
3. Export top speculative plans as composite maps and GIS-style layers.
4. Optionally produce Graph2City-compatible exports.


## Outline

1. Configure Boulder paths and run directory.
2. Run baseline build.
3. Generate ensemble plans.
4. Score and inspect Pareto/frontier outputs.
5. Export top plans and quick summaries.
6. Exercise: compare two Boulder profiles.
7. Pitfalls and extension ideas.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from civicmorph.pipeline import build_baseline, generate_ensemble, score_ensemble, export_top_plans

CITY = "boulder_co"
RUN_DIR = Path("runs") / f"{CITY}_tutorial"
DATA_DIR = Path("data") / "boulder"

OSM_PBF = DATA_DIR / "boulder.osm.pbf"
DEM = DATA_DIR / "boulder_dem.tif"
FLOOD = DATA_DIR / "boulder_flood.tif"
GRAPH2CITY_SEED = DATA_DIR / "graph2city_seed"

RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run directory:", RUN_DIR)
print("OSM exists:", OSM_PBF.exists())
print("DEM exists:", DEM.exists())
print("Flood exists:", FLOOD.exists())


## Step 1: Build baseline for Boulder

This creates baseline cells, terrain layers, and baseline graph placeholders used by downstream generation and scoring.


In [ ]:
if not OSM_PBF.exists():
    raise FileNotFoundError(f"Missing Boulder OSM input: {OSM_PBF}")

baseline = build_baseline(
    osm_pbf=str(OSM_PBF),
    dem=str(DEM) if DEM.exists() else None,
    flood=str(FLOOD) if FLOOD.exists() else None,
    project_dir=RUN_DIR,
)

print("Baseline metadata:")
print(json.dumps(baseline.metadata, indent=2))


## Step 2: Generate an ensemble

Use a Boulder-relevant profile and deterministic seed so results are reproducible.


In [ ]:
members = generate_ensemble(
    profile_name="optimistic_courtyard_city",
    ensemble_size=20,
    seed=1,
    project_dir=RUN_DIR,
)

print("Members generated:", len(members))
members.head(3)


## Step 3: Score plans (with optional ABM)

`--with-abm` behavior is represented here by setting `with_abm=True`.
If Mesa is not installed, run with `with_abm=False` or install `civicmorph[abm]`.


In [ ]:
try:
    scores = score_ensemble(project_dir=RUN_DIR, with_abm=True, abm_top=5, seed=1)
except Exception as exc:
    print("ABM scoring fallback to static-only:", exc)
    scores = score_ensemble(project_dir=RUN_DIR, with_abm=False, abm_top=5, seed=1)

scores = scores.sort_values("final_with_abm", ascending=False)
scores[["member_id", "static_final", "abm_penalty", "final_with_abm"]].head(10)


Interpretation:
- `static_final` is the baseline multi-objective score minus static penalties.
- `abm_penalty` captures post-plan regressions from agent simulation signals.
- `final_with_abm` is the decision rank metric when ABM is enabled.


## Step 4: Export top Boulder plans

Exports composite PNG, interactive HTML, and combined layer artifacts for presentation.


In [ ]:
exports = export_top_plans(project_dir=RUN_DIR, top_n=3, graph2city_out=False)
exports


In [ ]:
exports_dir = RUN_DIR / "exports"
summary_path = exports_dir / "ensemble_summary.md"
print("Summary exists:", summary_path.exists())
print(summary_path.read_text()[:600])


## Optional: Graph2City import/export for Boulder

If `data/boulder/graph2city_seed` and `civicmorph[graph2city]` are available:
1. Re-run generation with `seed_source="hybrid"` and `graph2city_in=...`.
2. Export top plans with `graph2city_out=True`.


In [ ]:
if GRAPH2CITY_SEED.exists():
    try:
        _ = generate_ensemble(
            profile_name="transit_corridor_city",
            ensemble_size=10,
            seed=2,
            project_dir=RUN_DIR,
            seed_source="hybrid",
            graph2city_in=str(GRAPH2CITY_SEED),
        )
        g2c_exports = export_top_plans(project_dir=RUN_DIR, top_n=2, graph2city_out=True)
        print("Graph2City exports ready:", g2c_exports)
    except Exception as exc:
        print("Graph2City path unavailable in this environment:", exc)
else:
    print("Skipping Graph2City step; seed package not found:", GRAPH2CITY_SEED)


## Exercise

Compare two Boulder profiles and answer:
- Which profile improves `green_access_score` most?
- Which profile has the best `final_with_abm` top-1 member?

Use the scaffold below.


In [ ]:
profile_names = ["green_weave_first", "transit_corridor_city"]
results = []

for profile in profile_names:
    run_dir = Path("runs") / f"boulder_exercise_{profile}"
    run_dir.mkdir(parents=True, exist_ok=True)

    _ = build_baseline(
        osm_pbf=str(OSM_PBF),
        dem=str(DEM) if DEM.exists() else None,
        flood=str(FLOOD) if FLOOD.exists() else None,
        project_dir=run_dir,
    )
    _ = generate_ensemble(profile_name=profile, ensemble_size=10, seed=3, project_dir=run_dir)
    s = score_ensemble(project_dir=run_dir, with_abm=False, seed=3)

    best = s.sort_values("final_with_abm", ascending=False).iloc[0]
    results.append(
        {
            "profile": profile,
            "top_member": int(best["member_id"]),
            "top_final_with_abm": float(best["final_with_abm"]),
            "mean_green_access": float(s["green_access_score"].mean()),
        }
    )

pd.DataFrame(results).sort_values("top_final_with_abm", ascending=False)


## Pitfalls and Extensions

Common pitfall:
- Comparing runs with different seeds and treating changes as policy effects rather than stochastic/sample effects.

Fix:
- Keep `seed` fixed when comparing profiles and only vary one profile/budget axis at a time.

Extension ideas:
- Add neighborhood-specific post-processing for Boulder subareas.
- Run ABM on larger `abm_top` sets to stress-test ranking stability.
- Build a dashboard app from `runs/boulder_demo/scoring/member_scores.parquet`.
